# Day 26 — 기본 feature 추출: 신호를 표로 바꾸기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-4/day-26.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-4/day-26.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1: 가상 참가자 12명으로 feature table 두 개 만들기

코드가 길지만 네 덩어리입니다. ① 가상 데이터 만들기, ② feature 함수, ③ 모든 기록에 적용, ④ 사람 단위로 접기. **실제 데이터가 아닙니다.** PD군에 떨림을 넣은 것은 코드를 연습하기 위한 설정일 뿐, 실제 군 차이에 대한 주장이 아닙니다.

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal as sg

FS = 100            # sampling rate (Hz)
N = 1024            # 기록 하나의 표본 수 (10.24초)
rng = np.random.default_rng(42)

# ---------- 1. 가상 데이터 만들기 (실제 데이터가 아님!) ----------
participants = pd.DataFrame({
    "participant_id": [f"S{i:02d}" for i in range(1, 13)],
    "group": ["PD"] * 6 + ["HC"] * 6,
})
# 사람마다 고유한 떨림 세기·주파수를 하나씩 뽑는다 → 같은 사람의 기록끼리 닮게 된다
participants["tremor_amp"] = np.where(participants["group"] == "PD",
                                      rng.uniform(0.01, 0.06, 12), rng.uniform(0.0, 0.01, 12))
participants["tremor_hz"] = rng.uniform(4, 6, 12)

def make_record(amp, hz, task):
    """가상 손목 가속도 한 축(단위 g)을 만든다."""
    t = np.arange(N) / FS
    scale = 1.0 if task == "Relaxed" else 0.5          # 가정: 쉬는 과제에서 떨림이 더 큼
    x = scale * amp * np.sin(2 * np.pi * hz * t)       # 떨림
    x = x + rng.normal(0, 0.01, N)                     # 센서 잡음
    x[:50] += 0.2 * np.exp(-np.arange(50) / 10)        # 시작 진동 알림 artifact (처음 0.5초)
    return x

# ---------- 2. feature 함수 ----------
def extract_features(x, fs=FS):
    x = x[int(0.5 * fs):]                  # 처음 0.5초 버리기 (artifact)
    x = x - np.mean(x)                     # 평균 제거 (중력·자세 성분)
    freqs, power = sg.welch(x, fs=fs, nperseg=256)
    band = (freqs >= 3) & (freqs <= 8)
    return {
        "std_g": np.std(x),                                   # 움직임 크기
        "range_g": np.max(x) - np.min(x),                     # 최대-최소
        "dom_freq_hz": freqs[1:][np.argmax(power[1:])],       # 가장 강한 주파수
        "band_3_8_frac": power[band].sum() / power[1:].sum()  # 3–8 Hz 비율
    }

# ---------- 3. 모든 기록에 적용 → record-level feature table ----------
rows = []
for p in participants.itertuples():
    for task in ["Relaxed", "StretchHold"]:
        for wrist in ["Left", "Right"]:
            x = make_record(p.tremor_amp, p.tremor_hz, task)
            # ↑↑↑ 실제 데이터로 바꿀 때: 이 한 줄을 "파일에서 신호 한 축 읽기"로 교체 ↑↑↑
            rows.append({"participant_id": p.participant_id, "group": p.group,
                         "task": task, "wrist": wrist, **extract_features(x)})
records = pd.DataFrame(rows)
print("record-level 표 크기:", records.shape)
print(records.head(4).round(3).to_string(index=False))

# ---------- 4. 참가자 단위로 요약 → participant-level feature table ----------
per_person = (records.groupby(["participant_id", "group", "task"])[["std_g", "band_3_8_frac"]]
              .mean().unstack("task"))
per_person.columns = [f"{feat}_{task}" for feat, task in per_person.columns]
per_person = per_person.reset_index()
print("\nparticipant-level 표 크기:", per_person.shape)
print(per_person.round(3).to_string(index=False))

assert per_person["participant_id"].is_unique          # 한 사람 = 한 줄인지 자동 확인
records.to_csv("pd_features_records.csv", index=False)
per_person.to_csv("pd_features_participants.csv", index=False)

### 예시 2: stride interval의 평균, SD, CV

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

# 가상 stride interval(한쪽 발이 땅에 닿고 다시 닿을 때까지의 시간, 초) 100걸음
steady = rng.normal(1.05, 0.02, 100)     # 규칙적으로 걷는 가상 인물 A
uneven = rng.normal(1.15, 0.06, 100)     # 걸음마다 들쭉날쭉한 가상 인물 B

for name, s in [("A", steady), ("B", uneven)]:
    mean, sd = s.mean(), s.std(ddof=1)
    print(f"{name}: 평균 {mean:.3f}초, SD {sd:.3f}초, CV {100 * sd / mean:.1f}%")

### 예시 3: 힘 신호에서 heel strike 찾기

In [ ]:
import numpy as np

fs = 100
t = np.arange(0, 20, 1 / fs)
# 가상 '한쪽 발 총 힘' 신호: 1.1초마다 한 번씩 발을 디딘다 (디딜 때만 힘 > 0)
force = np.clip(np.sin(2 * np.pi * t / 1.1), 0, None) * 600     # 단위 N

on_ground = force > 20                                  # 문턱값 20 N 이상이면 '디딤'
heel_strikes = np.where(~on_ground[:-1] & on_ground[1:])[0] + 1   # 공중→디딤으로 바뀌는 순간
stride_intervals = np.diff(heel_strikes) / fs           # 표본 개수 → 초

print("찾은 heel strike 수:", len(heel_strikes))
print("stride interval 처음 5개(초):", stride_intervals[:5])
print("평균:", round(stride_intervals.mean(), 3), "초")

## 직접 해보기

1. 새 노트북 `notebooks/05_pd_feature_analysis.ipynb`를 만들고 예시 1을 실행합니다. `extract_features`에서 `x = x[int(0.5 * fs):]` 줄을 주석 처리하고 다시 실행해 `range_g`가 얼마나 커지는지 확인한 뒤, 원래대로 되돌립니다.
2. **실제 데이터로 바꿔 끼우기.** Day 24에서 고른 데이터에 맞춰 "신호 한 개 읽기" 함수를 만듭니다. 열 번호와 이름은 **내 데이터 사전에서 확인한 것**을 씁니다.

In [ ]:
import pandas as pd

def load_signal(path, column):
    """파일 하나에서 열 하나를 numpy 배열로 읽는다. column은 0부터 세는 열 번호."""
    df = pd.read_csv(path, header=None, sep=None, engine="python")
    return df[column].to_numpy()

x = load_signal("<파일 경로>", <열 번호>)        # 자리표시자 두 곳을 채우기
print(len(x), x[:5])